In [1]:
from DS_8.config.utils import load_train_eval_climate_fever_data

df_train, df_eval = load_train_eval_climate_fever_data()

In [2]:
X_train = df_train["claim"]
y_train = df_train["claim_label"]

X_eval = df_eval["claim"]
y_eval = df_eval["claim_label"]

In [3]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import numpy as np

SOLVER_CHOICES = ["saga", "lbfgs", "newton-cholesky"]
NGRAM_RANGE_CHOICES = [(1,1), (1,2), (1,3)]
USE_IDF_CHOICES = [True, False]
SUBLINEAR_TF_CHOICES = [True, False]
NORM_CHOICES = ["l2", None]
search_space = {
    "C": hp.loguniform("C", np.log(1e-4), np.log(100)), # Regularization strength
    "solver": hp.choice("solver", SOLVER_CHOICES),
    "l1_ratio": hp.uniform("l1_ratio", 0.0, 1.0),
    "max_iter": hp.qloguniform("max_iter", np.log(200), np.log(2000), 100),
    "max_features": hp.qloguniform("max_features", np.log(500), np.log(20000), 500),
    "min_df": hp.qloguniform("min_df", np.log(1), np.log(20), 1),
    "max_df": hp.uniform("max_df", 0.6, 0.95),
    "ngram_range": hp.choice("ngram_range", NGRAM_RANGE_CHOICES),
    "sublinear_tf": hp.choice("sublinear_tf", SUBLINEAR_TF_CHOICES),
    "use_idf": hp.choice("use_idf", USE_IDF_CHOICES),
    "norm": hp.choice("norm", NORM_CHOICES),
}
PARAM_STEP_MAPPNG = {
    "max_features": "tfidf",
    "min_df": "tfidf",
    "max_df": "tfidf",
    "ngram_range": "tfidf",
    "sublinear_tf": "tfidf",
    "use_idf": "tfidf",
    "norm": "tfidf",
	"C": "clf",
    "penalty": "clf",
    "solver": "clf",
    "l1_ratio": "clf",
    "max_iter": "clf"
    }

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

def build_model(params:dict):
    param_dict_dict = {"tfidf":{}, "clf":{}}
    for p, v in params.items():
        param_dict_dict[PARAM_STEP_MAPPNG[p]][p] = v
    return Pipeline(steps=[
    ('tfidf', TfidfVectorizer(**param_dict_dict["tfidf"])),
    ('clf', LogisticRegression(
            class_weight="balanced",
            random_state=42,
            **param_dict_dict["clf"])
            )
        ])

In [5]:
PROD_MODEL_THRESHOLD = 0.5

In [7]:
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, classification_report

def make_objective(X_tune_train, y_tune_train, X_tune_test, y_tune_test):
	def objective(params):
		# Invalid combinations
		if params["solver"] != "saga" and "l1_ratio" in params:
			params["l1_ratio"] = None
			params["penalty"] = None
		params["max_features"] = int(params["max_features"])
		params["min_df"] = int(params["min_df"])
		params["max_iter"] = int(params["max_iter"])

		model = build_model(params)
		model.fit(X_tune_train, y_tune_train)

		# train_preds = model.predict_proba(X_tune_train)[:, 1]
		test_preds = model.predict_proba(X_tune_test)[:, 1]

		# train_auc = roc_auc_score(y_tune_train, train_preds)
		test_auc = roc_auc_score(y_tune_test, test_preds)
		# train_f1 = f1_score(y_tune_train == "SUPPORTS", train_preds > PROD_MODEL_THRESHOLD)
		# test_f1 = f1_score(y_tune_test == "SUPPORTS", test_preds > PROD_MODEL_THRESHOLD)
		# train_acc = accuracy_score(y_tune_train == "SUPPORTS", train_preds > PROD_MODEL_THRESHOLD)
		# test_acc = accuracy_score(y_tune_test == "SUPPORTS", test_preds > PROD_MODEL_THRESHOLD)

		# print("train_auc: ", train_auc)
		# print("test_auc: ", test_auc)
		# print("train_f1: ", train_f1)
		# print("test_f1: ", test_f1)
		# print("train_acc: ", train_acc)
		# print("test_acc: ", test_acc)
		# print("-------------")
		# plot_roc_curve(
		# 	y_tune_test,
		# 	test_preds,
		# 	title=f"ROC Curve - Tuning (AUC={test_auc:.4f})"
		# )

		return {
			"loss": 1 - test_auc,
			"status": STATUS_OK
		}
	return objective

In [8]:
def make_tuning_split(X, y, test_size=0.2, seed=42):
	from sklearn.model_selection import train_test_split
	return train_test_split(
		X, y,
		test_size=test_size,
		stratify=y,
		random_state=seed
	)

In [9]:
def hyperparameter_tuning(X_train, y_train):
	X_tune_train, X_tune_test, y_tune_train, y_tune_test = make_tuning_split(
		X_train, y_train
	)

	objective_fn = make_objective(
		X_tune_train, y_tune_train,
		X_tune_test, y_tune_test
	)

	trials = Trials()

	best_params = fmin(
		fn=objective_fn,
		space=search_space,
		algo=tpe.suggest,
		max_evals=100,
		trials=trials
	)

	# Cast params
	best_params["solver"] = SOLVER_CHOICES[best_params["solver"]]
	best_params["ngram_range"] = NGRAM_RANGE_CHOICES[best_params["ngram_range"]]
	best_params["use_idf"] = USE_IDF_CHOICES[best_params["use_idf"]]
	best_params["sublinear_tf"] = SUBLINEAR_TF_CHOICES[best_params["sublinear_tf"]]
	best_params["norm"] = NORM_CHOICES[best_params["norm"]]
	if best_params["solver"] != "saga" and "l1_ratio" in best_params:
		best_params["l1_ratio"] = None
		best_params["penalty"] = None
	best_params["max_features"] = int(best_params["max_features"])
	best_params["min_df"] = int(best_params["min_df"])
	best_params["max_iter"] = int(best_params["max_iter"])

	return best_params

In [10]:
def final_training(X_train, y_train, X_val, y_val, params):
	model = build_model(params)
	model.fit(X_train, y_train)

	train_preds = model.predict_proba(X_train)[:, 1]
	val_preds = model.predict_proba(X_val)[:, 1]

	train_auc = roc_auc_score(y_train, train_preds)
	val_auc = roc_auc_score(y_val, val_preds)
	train_f1 = f1_score(y_train == "SUPPORTS", train_preds > PROD_MODEL_THRESHOLD)
	test_f1 = f1_score(y_val == "SUPPORTS", val_preds > PROD_MODEL_THRESHOLD)
	train_acc = accuracy_score(y_train == "SUPPORTS", train_preds > PROD_MODEL_THRESHOLD)
	test_acc = accuracy_score(y_val == "SUPPORTS", val_preds > PROD_MODEL_THRESHOLD)
	
	print("train_auc: ", train_auc)
	print("test_auc: ", val_auc)
	print("train_f1: ", train_f1)
	print("test_f1: ", test_f1)
	print("train_acc: ", train_acc)
	print("test_acc: ", test_acc)
	print("\nClassification report:")
	print(classification_report(y_val == "SUPPORTS", val_preds > PROD_MODEL_THRESHOLD, target_names=["REFUTES", "SUPPORTS"]))
	
	plot_roc_curve(y_val, val_preds, title=f"ROC Curve - Final (AUC={val_auc:.4f})")

In [11]:
# ---------- Phase 1: Hyperparameter tuning ----------
best_params = hyperparameter_tuning(X_train, y_train)

# ---------- Phase 2: Final training ----------
final_training(X_train, y_train, X_eval, y_eval, best_params)

  1%|          | 1/100 [00:02<03:48,  2.31s/trial, best loss: 0.39141414141414144]

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1170: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(



  7%|▋         | 7/100 [00:02<00:25,  3.69trial/s, best loss: 0.3419612794612795] 

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:591: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
A singular matrix detected: slice(s) [0] are singular.
  warnings.warn(

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.wa

 10%|█         | 10/100 [00:02<00:16,  5.49trial/s, best loss: 0.3419612794612795]

...

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:591: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
An ill-conditioned matrix detected: slice 0 has rcond = 3.9063028766068207e-38.
  warnings.warn(

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of pe

100%|██████████| 100/100 [01:23<00:00,  1.20trial/s, best loss: 0.2767255892255893]

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:591: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
An ill-conditioned matrix detected: slice 0 has rcond = 4.588151385322672e-50.
  warnings.warn(

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of pen

c:\Applications\Git-OpenClassrooms\DS_8\.venv\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:591: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration 1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
An ill-conditioned matrix detected: slice 0 has rcond = 3.036890939067426e-39.
  warnings.warn(


train_auc:  0.9999617027795272
test_auc:  0.6732919254658385
train_f1:  0.9972093023255814
test_f1:  0.8105726872246696
train_acc:  0.9960369881109643
test_acc:  0.7133333333333334

Classification report:
              precision    recall  f1-score   support

     REFUTES       0.39      0.43      0.41        35
    SUPPORTS       0.82      0.80      0.81       115

    accuracy                           0.71       150
   macro avg       0.61      0.61      0.61       150
weighted avg       0.72      0.71      0.72       150

